## AOPC metric

In [1]:
import torch
import torch.nn.functional as F
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))
from torch_geometric.nn import GCNConv
from graphLoader import GraphLoader
import numpy as np
import time
from torch_geometric.logging import init_wandb, log
import gc 
import networkx as nx
from pyHSICLasso import HSICLasso
from collections import Counter

In [2]:
graph_dataset_name ="Cora"
embedding_dimension = 64
embedding_method = "GCN"

embeddings_path = f'./supervised_embeddings/{graph_dataset_name}/{embedding_method}_{embedding_dimension}.npy'
predictions_path = f'./AOPC/predictions/{graph_dataset_name}/{embedding_method}_{embedding_dimension}.npy'

graph_file_path = f'data/{graph_dataset_name}/{graph_dataset_name}_combined.txt'
graphLoader = GraphLoader(graph_file_path, graph_dataset_name)
G_nx, G_pyg = graphLoader.load_graph()
num_classes = torch.unique(G_pyg.y).numel()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
graph = G_pyg.to(device)

NetworkX - Number of nodes: 2485
NetworkX - Number of edges: 5069
NetworkX graph is undirected.
NetworkX graph is connected: True
Number of graphs: 6
PyG - Number of features per node: 1433
Number of classes: 7
PyG - Number of nodes: 2485
PyG - Number of edges: 10138
Is undirected: True
PyG - Number of features per node: 1433


In [3]:
def load_embeddings(embeddings_path):
    if os.path.exists(embeddings_path):
        print(f"Loading existing embeddings from {embeddings_path}")
        return np.load(embeddings_path)
    return None 
def save_embeddings(embeddings, embeddings_path):
    if embeddings_path is not None and embeddings_path:

        os.makedirs(os.path.dirname(embeddings_path), exist_ok=True)
        np.save(embeddings_path, embeddings)
        print(f"Embeddings saved to {embeddings_path}")
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels
                            )
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.out = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_weight=None, return_embeddings = False):
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv1(x, edge_index, edge_weight).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        
        embeddings = x
        out = self.out(embeddings)
        
        if return_embeddings:
            return embeddings
        return out

def train():
    model.train()
    optimizer.zero_grad()
    out = model(graph.x, graph.edge_index, graph.edge_attr)
    loss = F.cross_entropy(out[graph.train_mask], graph.y[graph.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss)

@torch.no_grad()
def test():
    model.eval()
    pred = model(graph.x, graph.edge_index, graph.edge_attr).argmax(dim=-1)

    accs = []
    for mask in [graph.train_mask, graph.val_mask, graph.test_mask]:
        accs.append(int((pred[mask] == graph.y[mask]).sum()) / int(mask.sum()))
    return accs


embeddings = load_embeddings(embeddings_path)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model_path = f"AOPC_saved_models/{graph_dataset_name}_{embedding_method}.pth"
os.makedirs(os.path.dirname(model_path), exist_ok=True)

model = GCN(
    in_channels=graph.num_features,
    hidden_channels=64,
    out_channels=num_classes,
).to(device)

model_exists = os.path.exists(model_path)
if model_exists:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded from {model_path}")
else:
    print("No saved model found. Training a new one.")

optimizer = torch.optim.Adam([
    dict(params=model.conv1.parameters(), weight_decay=5e-4),
    dict(params=model.conv2.parameters(), weight_decay=0)
], lr=0.01)

if embeddings is None or not model_exists:
    best_val_acc = test_acc = 0
    times = []

    for epoch in range(1, 200 + 1):
        start = time.time()
        loss = train()
        train_acc, val_acc, tmp_test_acc = test()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            test_acc = tmp_test_acc
        log(Epoch=epoch, Loss=loss, Train=train_acc, Val=val_acc, Test=test_acc)
        times.append(time.time() - start)

    print(f'Median time per epoch: {torch.tensor(times).median():.4f}s')
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

    with torch.no_grad():
        model.eval()
        embeddings = model(graph.x, graph.edge_index, return_embeddings=True).cpu().detach().numpy()
        save_embeddings(embeddings, embeddings_path)

with torch.no_grad():
    out = model(graph.x, graph.edge_index)
    all_predictions = out.argmax(dim=1).cpu().numpy()

    import pandas as pd
    predictions_df = pd.DataFrame({
        'Node Index': range(len(all_predictions)),
        'Predicted Class': all_predictions
    })
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading existing embeddings from ./supervised_embeddings/Cora/GCN_64.npy
Model loaded from AOPC_saved_models/Cora_GCN.pth


In [4]:
def get_2_hop_neighbors(graph, node):
    one_hop_neighbors = set(graph.neighbors(node))
    two_hop_neighbors = set()
    for neighbor in one_hop_neighbors:
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    return two_hop_neighbors

num_nodes = 200
nodes_with_large_2hop_neighbors = []
for node in G_nx.nodes():
    two_hop_neighbors = get_2_hop_neighbors(G_nx, node)
    if len(two_hop_neighbors) > 100:
        nodes_with_large_2hop_neighbors.append(node)
    if len(nodes_with_large_2hop_neighbors) >= num_nodes:
        break
print(f"Found {len(nodes_with_large_2hop_neighbors)} nodes with 2-hop neighborhoods greater than 100:")
print(len(nodes_with_large_2hop_neighbors))
output_path = "nodes_cora.txt"
with open(output_path, "w") as f:
    for n in nodes_with_large_2hop_neighbors:
        f.write(f"{n}\n")

print(f"Saved nodes to {output_path}")

Found 200 nodes with 2-hop neighborhoods greater than 100:
200
Saved nodes to nodes_cora.txt


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

def most_frequent(values):
    values = values.astype(int)  
    return np.bincount(values).argmax()

def compute_aopc_global_gcn(model, graph, feature_ranking, K):
    """
    Compute AOPC global for a GCN model.

    """
    device = next(model.parameters()).device  
    model.eval()  

    X = graph.x.clone().to(device)  
    edge_index = graph.edge_index.to(device)  

    N = X.shape[0]  
    cumulative_diffs = np.zeros(N)  

   
    with torch.no_grad():
        orig_probs = model(X, edge_index)  
        pred_classes = orig_probs.argmax(dim=1)    
        orig_class_probs = orig_probs[torch.arange(N), pred_classes].cpu().numpy()
         
    for k in range(K + 1):
        X_modified = X.clone()

        if k > 0:
            features_to_remove = feature_ranking[:k]
            for feature in features_to_remove:
                feature_agg_value = most_frequent(X[:, feature].cpu().numpy())  
                X_modified[:, feature] = feature_agg_value  

    
        with torch.no_grad():
            new_probs = model(X_modified, edge_index)
            new_class_probs = new_probs[torch.arange(N), pred_classes].cpu().numpy()

       
        cumulative_diffs += (orig_class_probs - new_class_probs)
    aopc = np.mean(cumulative_diffs) / (K + 1)
    return aopc

def plot_aopc_global_gcn(model, graph, experiments, K_max, random_runs=5):
    """
    Plot AOPC global curves for different feature ranking methods for a GCN model.
    
    """
    device = next(model.parameters()).device
    plt.figure(figsize=(10, 6))
    
    for method_name, feature_ranking in experiments.items():
        aopc_values = []
        for k in tqdm(range(K_max + 1), desc=f"Computing {method_name}"):
            aopc = compute_aopc_global_gcn(model, graph, feature_ranking, k)
            aopc_values.append(aopc)
        
        plt.plot(range(K_max + 1), aopc_values, label=method_name, marker='o')
    
    random_curves = []
    num_features = graph.x.shape[1]
    for _ in range(random_runs):
        ranking = np.random.permutation(num_features).tolist()
        
        aopc_values = []
        for k in tqdm(range(K_max + 1), desc="Computing Random baseline"):
            aopc = compute_aopc_global_gcn(model, graph, ranking, k)
            aopc_values.append(aopc)
        random_curves.append(aopc_values)
    
    random_mean = np.mean(random_curves, axis=0)
    random_std = np.std(random_curves, axis=0)
    
    plt.plot(range(K_max + 1), random_mean, label='Random', linestyle='--', color='gray')
    plt.fill_between(range(K_max + 1), random_mean - random_std, random_mean + random_std, color='gray', alpha=0.2)
    
    plt.xlabel('Number of Features Removed (K)', fontsize=25)
    plt.ylabel('AOPC Global', fontsize=25)
    plt.tick_params(axis='x', labelsize=25)
    plt.tick_params(axis='y', labelsize=25)
    plt.legend()
    plt.grid(True)
    plt.savefig(f'AOPC_{graph_dataset_name}_{embedding_method}_{embedding_dimension}_.pdf', format='pdf', bbox_inches='tight')
    plt.show()


In [6]:
def compute_structural_node_features(G, node):
    degree = G.degree[node]
    avg_neighbor_degree_dict = nx.average_neighbor_degree(G, nodes=[node]).get(node, 0)
    clustering_coefficient = nx.clustering(G, node)
    node_triangles = nx.triangles(G)[node]
    ppr = nx.pagerank(G, alpha=0.85, personalization={node: 1})
    ppr_std = np.std(list(ppr.values()))

    neighbors = list(G.neighbors(node))
    avg_neighbor_clustering = np.mean([nx.clustering(G, n) for n in neighbors]) if neighbors else 0

    return np.array([
        degree,
        avg_neighbor_degree_dict,
        clustering_coefficient,
        node_triangles,
        ppr_std,
        avg_neighbor_clustering
    ])

def get_node_features(G, node_attrs, dataset_name, save_dir="Node_Features"):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{dataset_name}_features.npy")
    columns_path = os.path.join(save_dir, f"{dataset_name}_columns.txt")
    
    
    if os.path.exists(save_path) and os.path.exists(columns_path):
        features = np.load(save_path)
        with open(columns_path, "r") as f:
            columns = f.read().strip().split("\n")
        node_ids = sorted(G.nodes())
        return pd.DataFrame(features, index=node_ids, columns=columns)

  
    feature_columns = [
        "Degree",
        "Avg Neighbor Degree",
        "Clustering Coefficient",
        "Node Triangles",
        "Personalized Pagerank Std",
        "Avg Neighbor Clustering",
    ]
    attr_columns = [f"{str(i)}" for i in range(0,node_attrs.shape[1])]
    all_columns = feature_columns + attr_columns

    features = []
    node_ids = sorted(G.nodes())

    for node in tqdm(node_ids, desc="Computing node features"):
        attr = node_attrs[node]
        combined = np.concatenate([attr])
        features.append(combined)

    features = np.array(features)
    np.save(save_path, features)

    with open(columns_path, "w") as f:
        f.write("\n".join(all_columns))

    return pd.DataFrame(features, index=node_ids, columns=all_columns)

### TACENR Approach

In [7]:
from explainer import Explainer
ratios_to_try = [
    (0.8, 0.2) 
]
model_type = "linear"

node_attributes = graph.x
num_of_features = node_attributes.shape[1]


num_nodes = graph.num_nodes

 

node_attrs = (graph.x.cpu().numpy() if graph.x is not None
                      else np.eye(graph.num_nodes, dtype=np.float32))

feature_names = [str(i) for i in range(0, node_attrs.shape[1])]
node_features_df = get_node_features(G_nx, node_attrs, dataset_name=graph_dataset_name)
N = G_pyg.num_nodes

for sim_ratio, dis_ratio in ratios_to_try:
    print(f"\n--- Testing ratio: {int(sim_ratio*100)}% similar, {int(dis_ratio*100)}% dissimilar ---")
    T = min(int(0.10 * N), 300)
    T = max(T, 20)                      
    T = min(T, N - 1)                    

    m = int(T * sim_ratio)              
    n = T - m                            
    m = max(1, m)
    n = max(1, n)
    mode = "weighted"
    problem = "supervised"
    
    weighting_method="gradients_only_for_ranking"
    all_pprs = []
    print(f"Using m={m}, n={n} (T={T} ~ {100*T/N:.1f}% of {N})")
    explainer = Explainer(G_nx, G_pyg, node_features_df,all_pprs, nodes_with_large_2hop_neighbors, feature_names, graph_dataset_name, embedding_method, embeddings, model, explanation_type="global", 
                                    node_attributes=True, structural_features = False, problem='regression', model_type=model_type, num_similar=m, num_dissimilar=n, 
                                    scale=False, parameter_tuning=True, enable_plots=False, contrastive=True, weighting_method=weighting_method, include_target_node_features=False, use_proximity= True)
    
    frequency_df, avg_importances_df, all_importances_df, average_metrics, per_node_metrics_df  = explainer.explain(mode = mode, problem = problem)
    


--- Testing ratio: 80% similar, 20% dissimilar ---
Using m=198, n=50 (T=248 ~ 10.0% of 2485)


Processing nodes:   0%|          | 0/200 [00:00<?, ?it/s]

Processing nodes: 100%|██████████| 200/200 [02:53<00:00,  1.15it/s]


In [ ]:
experiments = {}

folder_name = "FeatureRankingsAOPC"
os.makedirs(folder_name, exist_ok=True)    

file_path = os.path.join(folder_name, f"feature_ranking_{graph_dataset_name}_{embedding_method}_TACENR.txt")

df_sorted = avg_importances_df.reindex(
    avg_importances_df["Importance"].sort_values(ascending=False).index
)

feature_ranking = df_sorted["Feature"].tolist()
feature_ranking = [int(feature) for feature in feature_ranking]

with open(file_path, "w") as f:
    for feature in feature_ranking:
        f.write(f"{feature}\n")

experiments["TACENR"] = feature_ranking

### GraphLIME Approach

In [9]:
import networkx as nx

def get_k_hop_neighbors(G, node, k):
    subgraph = nx.ego_graph(G, node, radius=k)
    subgraph_nodes = list(subgraph.nodes())
    subgraph_nodes.sort()
    
    return subgraph_nodes, len(subgraph_nodes)

def get_node_features(data, nodes):
    return data.x[nodes].cpu().numpy()

In [ ]:
import os
count = 0
top_feature_counts = Counter()
feature_importances = Counter()

for node in tqdm(nodes_with_large_2hop_neighbors, desc='Processing nodes'):

    subgraph_nodes, num_nodes =  get_k_hop_neighbors(G_nx, node, 2)
    if num_nodes > 100:
        count+=1
        node_features = get_node_features(G_pyg, subgraph_nodes)
        
        node_features_df = pd.DataFrame(node_features, columns=[f'feature_{i}' for i in range(node_features.shape[1])])
        node_features_df['Node Index'] = subgraph_nodes
        
        merged_df = pd.merge(node_features_df, predictions_df, on='Node Index', how='inner')
        merged_df.reset_index(drop=True, inplace=True)
        
        X = merged_df.drop(columns=['Node Index', 'Predicted Class']).values
        y = merged_df['Predicted Class'].values
        hsic_lasso = HSICLasso()
        hsic_lasso.input(X, y, M=3, B=16)
        hsic_lasso.classification()

        selected_feature_indexes = hsic_lasso.get_index()
        selected_feature_scores = hsic_lasso.get_index_score()
        
        top_features = sorted(zip(selected_feature_indexes, selected_feature_scores), key=lambda x: -x[1])[:10]
        for feature_idx, score in top_features:
            feature_importances[feature_idx] += score

average_importances = {feature_idx: importance / count for feature_idx, importance in feature_importances.items()}

average_importances_df = pd.DataFrame(average_importances.items(), columns=['Feature Index', 'Average Importance'])

average_importances_df = average_importances_df.sort_values(by='Average Importance', ascending=False)

Processing nodes: 100%|██████████| 200/200 [01:27<00:00,  2.29it/s]


In [ ]:
sorted_feature_list = average_importances_df['Feature Index'].tolist()

experiments["GraphLIME"] = sorted_feature_list

file_path = os.path.join(folder_name, f"feature_ranking_{graph_dataset_name}_{embedding_method}_GraphLIME.txt")
with open(file_path, "w") as f:
    for feature in sorted_feature_list:
        f.write(f"{feature}\n")

### GNNExplainer Approach

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=50),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type=None,
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='raw',
    ),
)

model.eval()
all_feature_importances = []

for node_index in tqdm(nodes_with_large_2hop_neighbors, desc="Explaining nodes"):
    explanation = explainer(graph.x, graph.edge_index, index=node_index)
    
    feature_importances = explanation.node_mask
    node_feature_importances = feature_importances[node_index]
    node_feature_importances_np = node_feature_importances.cpu().detach().numpy()

    all_feature_importances.append(node_feature_importances_np)
all_feature_importances = np.stack(all_feature_importances) 
avg_feature_importances = np.mean(all_feature_importances, axis=0)  
ranked_features = np.argsort(-avg_feature_importances)  

feature_ranking = [(i, avg_feature_importances[i]) for i in ranked_features]
ranked_feature_indices = ranked_features.tolist()
experiments["GNNExplainer"] = ranked_feature_indices

file_path = os.path.join(folder_name, f"feature_ranking_{graph_dataset_name}_{embedding_method}_GNNExplainer.txt")
with open(file_path, "w") as f:
    for feature in ranked_feature_indices:
        f.write(f"{feature}\n")

Explaining nodes: 100%|██████████| 200/200 [00:49<00:00,  4.06it/s]
